In [7]:
import pandas as pd
import yfinance as yf

In [15]:
#INTITAL PORTFOLIO FILTER -
#1. needs to be company stock
#2. remove stocks not listed in us or canada
#3. remove stocks not in USD or Cad
#4. keep only stocks with avg volume>=5000 (oct 1,2025 to sep 30,2026)
#5. exclude months with <18 trading days when finding avg


#read the tickers
tickers = pd.read_csv("Tickers_Example.csv", header=None)
tickers = tickers.dropna()

eligible = []
eligible_currency = []

for i in range(len(tickers)):
    symbol = str(tickers.iloc[i, 0])
    valid = True

    try:                                   
        stock = yf.Ticker(symbol)
        info = stock.info

        #company stock?
        if info.get("quoteType") == "EQUITY":
            valid = True
        else:
            valid = False

        #listed in US/CANADA?
        valid_listing = False
        if info.get("market") == "us_market":
            valid_listing = True
        if info.get("market") == "ca_market":
            valid_listing = True
        if  valid_listing == False:
            valid = False

        #currency in usd/cad?
        valid_currency = False
        if info.get("currency") == "USD":
            valid_currency = True
        if info.get("currency") == "CAD":
            valid_currency= True
        if valid_currency == False:
            valid = False

        #exclude stocks with avgv volume <5000 (oct 1,2025 to sep 30,2026) and months with <18 trading days are excluded in avg
        if valid:
            hist = stock.history(start="2025-10-01", end="2026-10-01")
            hist.index = hist.index.strftime('%Y-%m-%d')
            dates = pd.to_datetime(hist.index)

            days_per_month = hist["Volume"].groupby([dates.year, dates.month]).count()
            volume_per_month = hist["Volume"].groupby([dates.year, dates.month]).sum()

            total_days = 0
            total_volume = 0
            for m in range(len(days_per_month)):
                if days_per_month.iloc[m] >= 18:
                    total_days = total_days + days_per_month.iloc[m]
                    total_volume = total_volume + volume_per_month.iloc[m]

            if total_days == 0:
                valid = False
            elif total_volume/total_days < 5000:
                valid = False

    except Exception:
        valid = False

    #ticker alr in list?
    already_added = False
    for j in range(len(eligible)):
        if eligible[j] == symbol:
            already_added = True

    if valid and already_added == False:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))

#download price data for eligible tickers
all_stock = yf.download(eligible, start="2025-10-01" , end="2026-11-28" , group_by="ticker")
all_stock.index = all_stock.index.strftime('%Y-%m-%d')

stock_data = {}

for i in range(len(eligible)):
    ticker = eligible[i]

    indiv_stock = all_stock[ticker].copy()        
    indiv_stock = indiv_stock.dropna(how="all")    

    stock_data[ticker] = indiv_stock

#convert tickers in usd to cad
rate = ['CADUSD=X']
fx_tickers = yf.Tickers(' '.join(rate))

exchange_rate = []
for i in fx_tickers.tickers:
    exchange_rate.append(fx_tickers.tickers[i].history(start="2025-10-01", end="2026-11-28").Close)

cadusd = exchange_rate[0]
cadusd.index = cadusd.index.strftime('%Y-%m-%d')

for i in range(len(eligible)):
    ticker = eligible[i]
    table = stock_data[ticker]

    if eligible_currency[i] == "USD":
        table["CADUSD"] = cadusd
        table["Close_CAD"] = table["Close"] / table["CADUSD"]
    else:
        table["Close_CAD"] = table["Close"]

    stock_data[ticker] = table

print("Eligible tickers:", eligible)

for ticker in eligible:
    print("Comapny:", ticker)
    print(stock_data[ticker][["Close", "Close_CAD"]])


[*********************100%***********************]  35 of 35 completed


Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Comapny: AAPL
Price            Close   Close_CAD
Date                              
2025-10-01  254.316605  354.097724
2025-10-02  255.989151  356.836086
2025-10-03  256.875183  358.620883
2025-10-06  255.551132  356.877160
2025-10-07  255.342056  356.100050
...                ...         ...
2026-11-12  354.880005  505.629491
2026-11-13  356.269989  508.076658
2026-11-16  352.779999  503.371199
2026-11-17  351.910004  503.125720
2026-11-18  352.450012  503.971768

[286 rows x 2 columns]
Comapny: ABBV
Price            Close   Close_CAD
Date                              
2025-10-01  235.580231  328.010134
2025-10-02  228.041809  317.878888
2025-10-03  225.487274  314.800536
2025-10-06  221.901215  309.885049
2025-10-0